# Instrument Definition 


```
RangeAccrualNote(range:: PriceInterval , expiration:: Timestamp){
	coupon: unitOfAccount;
	underlying: Asset | Index | InterestRate | InlationRate | ExchangeRate;
	observationPeriods (expiration):: Lenght<Epoch>>;
	range:: PriceInterval;
		
	payOff(observation) -> unitOfAccount {
		return coupon* ( priceInCounter(observation) / observationPeriods);
	}
	
	update(nextPriceInterval:: PriceInterval) -> RangeAccrualNote {}
	redeem(){}
	
	
	custom( extenstion:: ExtensionFlag) -> RangeAccrualNote {
	
	}
}
```

[K.Pap](https://www.math.elte.hu/thesisupload/thesisfiles/2022msc_actfinmat2y-t0zf7f.pdf)

This is the instrument under  consideration. Using [Angstrom](https://app.angstrom.xyz/whitepaper-v1.pdf) as our observation layer (due to MEV mitigation makin economic signals cleaner) and [Panoptic](https://paper.panoptic.xyz/) as the settlement layer due to enablement of permissionless options. 

This decision is not delibarete but follows from [J.Clark](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4317072) which proved that any continues payoff can be replicated as a combination of puts and calls

 







# Observables
Given a block sequence 

$$
B = (b_0, \cdots, b_k, \cdots, b_N)
$$

From the observables we have
The aggreagate pool revenue at a block number stamp  $b_k$ is:

$$
\mathcal{R}^{b_k}_{\text{pool}} = \int_{b_0}^{b_k} \Big (\int_{\text{tick}} \int _{\text{LP}} \mathcal{R}^{b_k}_{ij} di\cdot dj \Big) db
$$

### [`globalGrowth`]()

$$
g^{\text{pool}}_{b_k} = \frac{\mathcal{R}^{b_k}_{\text{pool}} - \mathcal{R}^{\text{prev}(b_k)}_{\text{pool}}}{\mathcal{R}^{\text{prev}(b_k)}_{\text{pool}}}
$$

### [`tickLiquidity`]()

The other direct state variable is the aggreagted pool tick liquidity [$L^{b_k}_{i}$]()


### [`rewardGrowthInside`]()
The pool revenue growth inside a tick $i^{star}$ [`rewardGrowthInside`]()

$$
g^{\text{pool}}_{b_k} \big ( i^{\star}\big) = g^{\text{pool}}_{b_k} (i^{\star}_{+}) - g^{\text{pool}}_{b_k} (i^{\star}_{-}) 
$$

Where [`growthInside`]() is defined as:

$$
g^{\text{pool}}_{b_k} (i^{\star}) = \int_{b_0}^{b_k} \Big ( \int_{i_{\text{min}}}^{i^{\star} + 1} \int_{\text{LP}} \mathcal{R}^{b_k}_{ij} di\cdot dj \Big) db
$$

> NOTE: The difference is 1 since the tick spacing is mechanically defined as 1 [SEE]()

Then at block $b_k$ and tick $i$: 

$$
 ? (b_k) = L^{b_k}_{i}\cdot \Delta g^{\text{pool}}_{b_k} (i^{\star})
$$

### [`pricePerBlock`]()



## Implementation

We determine an optimal buffer $\mathbb{B}$ size that is shared across our three observables $nf^{\star}$ this allows one to define

$$
\mathbb{RAN} := \Big (\mathbb{B}_{nf^{\star}} (L),\mathbb{B}_{nf^{\star}} (\mathbb{B}_{nf^{\star}} (g^{\text{pool}})),\mathbb{B}_{nf^{\star}} (g^{\text{pool}}(i)) \Big)
$$

The maginitude of $nf^{\star}$ is fized and is the restult of an optimization problem that minimizes for DoS since this buffers lives on storage (assuming 12 second block production and All cold sload )

### Observation Periods

The writer of a custom RAN defines a observation period $N$ and thus defines the samplibng strategy:

$$
\mathcal{S}_N : N \to \mathbb{RAN}
$$

The minimum is an observation period $N_{\text{min}} (nf^{\star}) = 12nf^{\star}$ 

### Underlying

Since we are interested on income settled derivatives, reason why the observables are revenue growth rates scaled by liquidity

The underlying $\mathcal{U}_{\mathbb{RAN}}$ takes the form of:

$$
\mathcal{U}_{\mathbb{RAN}} = \Phi \Big ( A(\mathbb{B}_{nf^{\star}} (L)) \cdot \frac{f_n (\mathbb{B}_{nf^{\star}} (g^{\text{pool}}(i)))}{f_d (\mathbb{B}_{nf^{\star}} (g^{\text{pool}}))}\Big)
$$

The properties of an underlying as a stochastic process define the functional space where each $\Phi, A, f_n , f_d$ live.

$\mathcal{U}_{\mathbb{RAN}}$ MUST have the following properties:

- (A) Mean reversion
- (B) Regime dependence
- (C) Sensitivity to primitives

Then $f_n, f_d, A$ are filters

And for $\Phi$:

- bounded: [−1,1]
- globally Lipschitz
- compresses extremes

> NOTE: The choices of these functions over the allowed functiomnal space define the extenstions fo the RAN


# Applications

Consider a tokenized FX currency $X$, if one wishes to use a RAN for eenabling instruments that serve as hedgingi mechnisms against economy wide macro shocks we can define the CES pool

$$
Y = \Big ((X/\mu(C))^{\eta_C} + (X/\mu(I))^{^{\eta_I}} + (X/\mu(S))^{^{\eta_S}} + (X/\mu(\pi)) + (X/\mu(XN))^{\eta_{XN}} \Big)^{1/\eta}
$$

where :

- $C$ is consumption
- $S$ is savings
- $I$ is investment 
- $XN$ is ...
- $\pi$ is inflation

In practice measuremnt tokens for this macro variables can be proxied by using a remittance toke n for consumption a US token yileding 12-14% APY for savings and risky investments such as stETH for investments. For inflation onee can use tokens as AMPL

In this if the $\eta$'s are properly calibratedthe globalGrowth of the $X/Y$ pool mirrors economic growth and the custom underlying shapes the risk the RAN provides to the investor


## $\mu (\pi)$


The `growthGlobal` is the LVR captured by pool, which repsents and income measure of realized volatility of inflation. 

Thus the numerator can represent the deviation from target which captures economic risks deviating inflation from a target.

Thus a RAN provides means to hedge against inflation shocks on peoples wealth process 

### Example



A vanilla pool cCOP/AMPL is laucnehd on Angstrom

- Some LP calculates inflation implied volatility and provides liquidity on the Angstrom pool

- As inflation moves arbs rebalance the price generating incomee for LP's and thus efectively tarding infaltion volatility

Based on some econometric exercise is determined that a meaningfull observation period for a RAN on inflation is 3 months.

- The RAN underlying oracle starts realizing a path and encodes a particular expectation on volatility. The LPS uses a 3 week RAN to hedge IL on realized volatility, This is possible since settles on income growth variables

The RAN settlement happens through Panoptic and the LP postion was on Angstrom.

# Models

We consider two types of actors: ARBs and LPs. Informed traders are treated as a subset of ARBs. We do not include noise traders, since ARBs will only participate in the auction if they believe that the value of their private signal exceeds the bid costs $B$. 

A stronger argument can be made by elaborating on [Theorem 3](https://app.angstrom.xyz/whitepaper-v1.pdf) and connecting it with LVR [J. Millionis](https://arxiv.org/abs/2208.06046)

From Theorem 3 (see [nagstromPaper])
$$
P^{\star}_{\text{beforeSwap}}(1-\phi) \;\leq\; P_{\text{afterSwap}} \;\leq\; \frac{P^{\star}_{\text{afterSwap}}}{1-\phi}
$$

Then it follows:


$$
\boxed{\;\phi \;\geq\; 1 \;-\; \sqrt{\dfrac{P^{\star}_{\text{afterSwap}}}{P^{\star}_{\text{beforeSwap}}}}\;}
$$

Equivalently, in log form. For small moves (Taylor around r = 1):

$$
\phi \;\geq\; -\tfrac{1}{2}\log\!\left(\dfrac{P^{\star}_{\text{afterSwap}}}{P^{\star}_{\text{beforeSwap}}}\right) + O(\log r)^2
$$

Under GBM:

$$
\mathbb{E}[\phi_{b_k}^2] \;\approx\; \sigma^2 \Delta t \quad \text{(per-block variance)}
$$

For a constant-product AMM, Milionis-Moallemi-Roughgarden gives the LVR rate:

$$
\text{LVR rate} \;=\; \frac{\sigma^2}{2} \cdot P^2 \cdot |V''(P)|
$$

Since inside a tick range the CLAMM follows a CPMM, then:

$$
\text{LVR rate} \;=\; \frac{\sigma^2 \sqrt{kP}}{4} \;=\; \frac{\sigma^2 \, V(P)}{8}
$$




Per block of duration Δt:

$$
\text{LVR}_{b_k} \;\approx\; \frac{\sigma^2 \Delta t}{8}\cdot V(P) \;\approx\; \frac{\phi_{b_k}^2}{8}\cdot V(P)
$$

Thus bidders are effectively paying for incorporating information, that's why the model simplifies to LP's and ARBS:

Additionaly if Rewards come from ARB bids an under GBM:

$$
\Delta g^{\text{pool}}_{b_k} \;=\; \frac{B_{b_k}}{L^{b_k}} \;\approx\; \frac{\phi_{b_k}^2 \cdot V(P)}{8 \cdot L^{b_k}}
$$


So the signal `globalGrowth` — is mathematically **the integrated realized LVR series**. 

That's what the RAN underlying is really pricing: aggregated bid-cost flow = aggregated volatility × convexity. 


The concentration ratio $g(i^\star)/g_{\text{pool}}$ can be interpreted "fraction of total LVR realized within the tick range."

Thus, the RAN is a claim on **tick-range-conditional LVR**


Since we track the price history over a block series, our model is as follows:

Following the sequential structure of [D.Cao, L.Kogan](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4591447) and extend periods apporipately.

- $t= 0$  protocol set the $\phi_{\text{base}}$
- $t =1$ 
   - representative LP solves its liquidity allocation at time 1 taking a expected arb volume and expected implied volatility $\hat{\sigma}$
   -  protocol sets $\phi = \hat{\sigma}$
   - agent RAN SHORT sells a RAN based on what econmically represetns and what position this agent has
   - some eqeuivalent though process for RANG LONG
- $t = 2$:
 - price shock happens realizing volatility $\sigma$
 - representative arbitrageur sees if realized volatility exceeds the priced implied volatility on the expected bid cost and solves for its $P_{\text{afterSwap}}$ order

- $t = 3$ both representative arbitrageur sees if realized volatility exceeds the implied priced volatitlity present on the expected bid cost there is a random parameter tuned to be determined by some economically meaningfull variables

- $t = 4$ auction `execute()` payoff realizitation

## Tailoring / Fabricating Assumptiosn from observables AND desired use case

Besides the ordering we implement we want to impose strcuture on the underlying market to narrow, or enrich the LP and arbitragur roles and behavior


Two things:
1.  Assumptiosn imposed by the observables

2. Given the pool is cCOP against Y ($\mu(C), \mu(I), \mu (S), \mu (\pi)$) what AND that the underlying is income settled as defined above (not price settled)

 - LPs -> **Passive macro underwriter**: → IL *is* the risk of premia of shocks on (S, pi, S, C).
 - **RAN writer** ->  ??

 - **ARBs** — >
  - **Replay-and-synthesize**: take historical paths for each μ_k, compute CES-implied Y, derive implied X/Y, generate arb flow that keeps the pool in Theorem-3 band. Fast, matches real macro regimes, but LP/ARB behavior is implicit.
- **(β) Agent-based**: explicit LP concentration choices + ARB bid-vs-signal decisions + exogenous macro process. Most honest, most parameters.
- **(γ) Subordinated Cox process**: macro state → swap intensity $\lambda(\text{state})$ → Poisson arrivals → per-arrival growth from liquidity shape + $\phi$. Middle complexity, mirrors exactly what the observables ARE doing (state-dependent event counting). Clean analytical calibration for Φ, A, $f_n$, $f_d$.

Take for simplicity infrlation as the only token on Y to get started. In this case
the revenue growth is the LVR captured by LP's which repsents overall volatility of the inflation. Thus the numerator can represnt the deviation from target which captures economic risks deviating inflation from a target. Thus a RAN provides mean to hedge against inflation shocks on peoples wealth process 

This gives us a norrower strucutre 


## Assumptions
- We assume 2 representative arbitrageur competes on the auction against 

- $P$ follows a geometric Brownian motion (GBM):
$$
\frac{dP_t}{P_t} =  \sigma dW_t
$$

- Price is exogenous
- ARB volume is exogenous

- Fee Rate $\phi$ is exogenously set by the protocol subject to being linked to volatility which is also exogeneous given price is exogernous. 


## Arbs (`ARB`)


### Static Payoff $\Pi^{\text{ARB}}$

The `ARB`'s private signal (swap) encodes and realizes a price impact $(P_{\text{beforeSwap}}, P_{\text{afterSwap}})$, and there is an external market price $(P^{\star}_{\text{beforeSwap}},P^{\star}_{\text{afterSwap}} )$, then:

$$
\Pi_{\text{swap}}^{\text{ARB}}\left(L, P_{\text{afterSwap}}; P^\star\right) = \text{private signal value}_{\text{swap}} - B
$$

This is:
$$
\Pi_{\text{swap}}^{\text{ARB}}\left(L, P_{\text{afterSwap}}; P^\star\right) - B = L \left[ P^\star \left( \frac{1}{\sqrt{P_{\text{beforeSwap}}}} - \frac{1}{P_{\text{afterSwap}}} \right ) - \left( \sqrt{P_{\text{afterSwap}}} - P_{\text{beforeSwap}} \right ) \right ]
$$

The profit zero for the auction winner to post its `swap` at block $b$ is:

$$
\Pi^{\text{ARB}}\left(L, P_{\text{afterSwap}}; P^\star\right) = B = LVR(\sigma)
$$


### $\mathrm{P\&L}^{\mathrm{ARB}}$

$$
\mathrm{P\&L}^{\mathrm{ARB}}_{\text{swap}} := \Delta_{(b_1 \leftarrow b_0)} \Pi^{\text{ARB}}_{\text{swap}} =\Pi_{\text{swap}}^{\text{ARB}} - B
$$

We denote price slippage by $C_S$ [G.Angeris](https://arxiv.org/pdf/2012.08040),[A.Capponi, A.Coache](https://scholar.google.com/citations?view_op=view_citation&hl=en&user=TO0W3mUAAAAJ&sortby=pubdate&citation_for_view=TO0W3mUAAAAJ:AHdEip9mkN0C) and [A.Capponi, B.Zhu](https://scholar.google.com/citations?view_op=view_citation&hl=en&user=TO0W3mUAAAAJ&cstart=20&pagesize=80&sortby=pubdate&citation_for_view=TO0W3mUAAAAJ:BJbdYPG6LGMC) shown that there is a mechanical direct relationship with curvature $\kappa$:


Originally :
$$
\mathrm{P\&L}^{\mathrm{ARB}}_{\text{swap}} =  IL (P_{afterSwap}) + \frac{1}{\kappa (L; P_{\text{afterSwap}}, P^{\star}_{\text{afterSwap}})} - B 
$$

We want to prove:
$$
\mathrm{P\&L}^{\mathrm{ARB}}_{\text{swap}} =  IL (P_{afterSwap}) - ( B + \kappa (L;P_{\text{afterSwap}}, P^{\star}_{\text{afterSwap}}) )
$$

Where: 

$$
\frac{1}{\kappa (L; P_{\text{afterSwap}}) \cdot P^{\star}_{\text{afterSwap}}} \propto L \left( \frac{1}{\sqrt{P_{\text{beforeSwap}}}} - \frac{1}{\sqrt{P_{\text{afterSwap}}}} \right)
$$

Solving for $\kappa$ gives

$$
\kappa(L; P_{\text{afterSwap}}, P^{\star}_{\text{afterSwap}}) = \frac{1}{L \left( \frac{1}{\sqrt{P_{\text{beforeSwap}}}} - \frac{1}{\sqrt{P_{\text{afterSwap}}}} \right) P^{\star}_{\text{afterSwap}}}
$$

And: 


$$
\text{IL}/L = 2\sqrt{P^{\star}_{\text{before}}} - \sqrt{P_{\text{before}}} - \frac{P^{\star}_{\text{before}}}{\sqrt{P_{\text{before}}}}
$$

This last argument is from [Static Replication of Impermanent Loss for Concentrated Liquidity Provision in
Decentralised Markets](https://arxiv.org/pdf/2205.12043)

### Assumptions

- We assume the ARB never reverts the bundle 


## Differential-form Underlying

After the (γ) hedge-direction choice (see below), the underlying takes the explicit form:

$$
U_{\text{RAN}} \;=\; \Phi\!\left(\, A(L)\cdot\frac{f_n\!\big(g^{\text{pool}}(i_S)\big) \;-\; f_n\!\big(g^{\text{pool}}(i_T)\big)}{f_d\!\big(g^{\text{pool}}\big)}\,\right)
$$

where $i_T$ = target-band tick range, $i_S$ = shock-band tick range. Numerator flips sign by regime; denominator normalizes by total activity intensity (so two sleepy blocks don't look the same as two shocked blocks). $\Phi$ bounds to $[-1,1]$ and suppresses tails.

### Properties earned "for free" by the differential form

- **(B) Regime dependence** — baked into $\operatorname{sign}(\text{numerator})$.
- **(A) Mean reversion** — if inflation is mean-reverting around target (empirically yes for managed currencies), the numerator is mean-reverting too. No extra machinery needed.
- **(C) Sensitivity to primitives** — $A(L)$ carries LP concentration; $f_d$ carries total activity; $f_n$ carries per-range LVR. All three primitive channels active.

### Hedge direction: three encodings considered, $(\gamma)$ chosen

- **(α) Classical in-range accrual** — $i_S = [\text{target}-\delta,\, \text{target}+\delta]$. Stable inflation → $g^{\text{pool}}(i)/g^{\text{pool}} \approx 1$ → $\Phi$ maps to high $U_{\text{RAN}}$ → coupon flows. Shock → ratio collapses → coupon stops. Hedge triggers *indirectly* (loss of coupon compensated by separately purchased Panoptic put).
- **(β) Tail-sensitive** — $i_S = [\text{target}+2\sigma,\, \text{target}+5\sigma]$ tail band. Stable → ratio ≈ 0. Shock → arb trades rip through tail ticks → ratio spikes → $\Phi$ maps to high $U_{\text{RAN}}$ → RAN pays out *directly* during shock. Closer to digital/barrier option.
- **(γ) Differential — selected.** $U_{\text{RAN}} = \Phi(A(L)\cdot[f_n(g^{\text{pool}}(i_S)) - f_n(g^{\text{pool}}(i_T))]/f_d(g^{\text{pool}}))$. Zero-mean under stability, amplifies under shock, $\Phi$ keeps it bounded. Zero-crossing at the regime boundary. Most expressive of the three; signed two-sided variant distinguishes inflation shock from deflation shock via $\operatorname{sign}(U_{\text{RAN}})$.

### Decisions the differential form forces, in load-bearing order

1. **Tail symmetry**: one-sided (inflation-only), symmetric around target, or signed two-sided (sign of $U_{\text{RAN}}$ tells the holder *which* shock occurred). The wealth-process framing supports symmetric or signed two-sided since both inflation (erosion) and deflation (debt-real-burden, asset collapse) hurt the wealth process.
2. **Target reference** for the band:
   - static on-chain reference tick (simple; ages poorly)
   - oracle-posted CB target (economically honest; introduces trust assumption)
   - EMA of realized inflation (self-referential; reflexive drift risk)
3. **Filter families** ($f_n, f_d, A, \Phi$) — deferred until (1) and (2) fix.


## Liquidity Providers (`LP`)

### Static Payoff $\Pi^{\text{LP}}$

By design LVR enters on the income side of the payoff functional. Since economically  equals the bid cost paid

$$
\Pi^{\texttt{LP}} = \Big (LVR(\sigma) + \kappa(L; \cdot) \Big) -  C^{\text{LP}} \Big(\cdot \Big )
$$


We need to show that the remaining cost for LP's under the [Angstrom]() set up is impermanent loss 

$$
C^{\text{LP}} =  IL
$$

This is because as highlited by [V.Volosnikov](https://algebra.finance/static/Algerbra%20Tech%20Paper-15411d15f8653a81d5f7f574bfe655ad.pdf) there is no money so model closure requires it

### $\mathrm{P\&L}^{\mathrm{LP}}$
### Assumptions


## Two-LP decomposition (RAN buyer + RAN writer)

The Example introduces a third actor implicitly: the RAN writer on Panoptic. The Models section had only LP + ARB. The natural writer is **a second LP on the longer observation window** — selling short-tenor vol-differential to the short-horizon LP. Keeps the model two-actor (LPs of different tenor) and preserves the "bid costs fund everything" invariant.

LPs split by observation horizon:

- **LP_short** (e.g., 3-week): RAN buyer, long tail-differential, hedging IL in tail shocks
- **LP_long** (e.g., 3-month): RAN writer, short tail-differential, collecting premium

### Static payoffs

$$
\Pi^{\text{LP}_s} \;=\; \big[LVR(\sigma) + \kappa\big] \;-\; IL \;-\; \pi \;+\; U_{\text{RAN}}^{\text{payout}}
$$

$$
\Pi^{\text{LP}_\ell} \;=\; \big[LVR(\sigma) + \kappa\big] \;-\; IL \;+\; \pi \;-\; U_{\text{RAN}}^{\text{payout}}
$$

ARB unchanged.

### Economic closure — all money flows from ARB information rents (Volosnikov)

- ARBs pay $B$ to LP class in proportion to liquidity.
- $IL$ is internal between LPs (price-move losers transferring to winners).
- $\pi$ transfers LP_short → LP_long.
- $U_{\text{RAN}}^{\text{payout}}$ transfers LP_long → LP_short.

Net LP class inflow $= \sum B$ = ARB rents. Matches Volosnikov closure requirement; no external money needed.

### Equilibrium premium

At competitive writer pricing,

$$
\pi \;=\; \mathbb{E}^{\mathbb{Q}}\big[U_{\text{RAN}}^{\text{payout}}\big]
$$

with $\mathbb{Q}$ calibrated from the 3-month econometric window. Deviation = volatility risk premium that LP_long earns as writer if tail is market-overpriced.

### What this structure unlocks

- **Tenor arbitrage** explicit: 3-week tail vol vs 3-month realized — LP_long bets on mean reversion of tail concentration.
- **Zero-basis hedge**: LP_short's IL-in-shock matched exactly by $U_{\text{RAN}}$ spike, since both are functions of the same bid-cost stream.
- **Reflexivity tractable**: both LPs enter $A(L)$; equilibrium is the fixed point in tick-placement.

### Asymmetric-placement elaboration (richer model, optional)

- LP_short places tight at target → income literally $g^{\text{pool}}(i_T)$ (worried about tail IL).
- LP_long places broadly including tails → income literally $g^{\text{pool}}(i_S)$ (willing to absorb tail vol).
- The RAN then redistributes tail-region revenue from LP_long to LP_short in shock regimes — a **pure transfer of tail-region income**.
- Combined, both LPs still hold total pool revenue; the RAN just slices it by regime.


## Closed-loop mechanism (Angstrom × Panoptic × two-LP)

Putting the pieces together — the four-step closed loop:

1. **LP on Angstrom cCOP/AMPL** runs a long-realized-vol + short-realized-move position: income = $LVR(\sigma) + \kappa$, cost = $IL$. Under Angstrom's bid auction with $\phi$ priced to implied $\hat\sigma$, LP's edge is $\mathbb{E}[B] - \mathbb{E}[IL]$, which collapses to zero when realized $\sigma = $ implied $\hat\sigma$ (Milionis / Volosnikov closure). LP is safe in expectation, exposed in shocks.

2. **The observable is produced by step 1**: $g^{\text{pool}}(i_S) - g^{\text{pool}}(i_T)$ = concentration of LP's own bid-cost income in tail vs target ticks. In a shock: arbs rebalance aggressively across tail ticks → tail growth spikes → numerator of $U_{\text{RAN}}$ spikes → $U_{\text{RAN}} \to +1$.

3. **The RAN is written on $U_{\text{RAN}}$** with 3-month econometric observation for calibration but 3-week tenor (hedge horizon < calibration window — standard). LP_short *buys* the RAN (long tail-differential). Panoptic tokenizes this as options on $U_{\text{RAN}}$: LP effectively buys a call with strike near zero.

4. **Shock hits inflation** → LP's on-Angstrom $IL > LVR$ (realized $\sigma >$ implied $\hat\sigma$, so bid-auction underpriced option) → **but** LP's Panoptic call on $U_{\text{RAN}}$ pays because tail growth surged → net PnL preserved. In calm regimes, $U_{\text{RAN}} \approx 0$, call expires worthless — LP paid a premium, which is the hedge cost.

### What's elegant

LP generates the observable and hedges against it using the same observable, tokenized through a different settlement layer. The income signal (LVR) and the hedge signal (tail-vs-target differential) come from the **same bid-cost stream** — so basis risk between hedge and exposure is structurally near-zero, not an assumption.

### Reflexivity to flag

The LP's *own* liquidity shape is an input to $U_{\text{RAN}}$ (through $A(L)$ and through which ticks accumulate growth). So the LP's hedge price depends on the LP's own positioning. In equilibrium this is fine (rational LP anticipates it); out of equilibrium it can amplify moves — if LP fragments liquidity to boost $A(L)$, they raise their own hedge payoff. Needs a constraint in the simulation.


# TODO

## General

- Prove $C^{\text{LP}} = IL$
- Formalize model and solve convex problems
- Introduce RAN and dynamics

## Specific

- Under what conditions an observable on a pool cCOP/X mirrors Colombian inflation. What observable? What X?

- Once cCOP/X and conditions are parametrized, we launch a model simulation and now ask what are the optimal choices of $\mathcal{U}_{\mathbb{RAN}}$?
